# Régression linéaire — données du professeur

Ce notebook charge le fichier CSV associé, ajuste une régression linéaire, évalue le modèle sur un jeu de test et visualise la droite et les résidus.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

In [ ]:
DATA_DIR = Path("data") if Path("data").is_dir() else Path("../data")
data_path = DATA_DIR / "Linear-regression-example-data.csv"

# Le fichier utilise un point-virgule comme séparateur et une virgule décimale.
data = pd.read_csv(data_path, sep=";", decimal=",")
data = data.apply(pd.to_numeric, errors="raise")

if list(data.columns) != ["X", "Y"]:
    raise ValueError("Le fichier doit contenir exactement les colonnes X et Y.")
if data.isna().any().any():
    raise ValueError("Le fichier contient des valeurs manquantes.")

print(f"{len(data)} observations chargées")
display(data.head())

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(data["X"], data["Y"], alpha=0.7, s=28)
plt.xlabel("X")
plt.ylabel("Y")
plt.title("Données de régression linéaire")
plt.grid(alpha=0.2)
plt.show()

In [ ]:
X = data[["X"]]
y = data["Y"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

model = LinearRegression()
model.fit(X_train, y_train)

sign = "+" if model.coef_[0] >= 0 else "-"
print(f"Équation estimée : Y = {model.intercept_:.4f} {sign} {abs(model.coef_[0]):.4f} × X")

In [ ]:
predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

metrics = pd.Series(
    {"MAE": mae, "RMSE": rmse, "R²": r2},
    name="Valeur",
).to_frame()
display(metrics.style.format("{:.4f}"))

if r2 <= 0:
    print("Conclusion : une droite n'explique pas utilement la variation de Y dans ces données.")
else:
    print(f"Conclusion : la droite explique {r2:.1%} de la variance observée sur le jeu de test.")

In [ ]:
x_line = pd.DataFrame(
    {"X": np.linspace(data["X"].min(), data["X"].max(), 200)}
)
y_line = model.predict(x_line)

plt.figure(figsize=(7, 5))
plt.scatter(X_train["X"], y_train, label="Entraînement", alpha=0.65, s=28)
plt.scatter(X_test["X"], y_test, label="Test", alpha=0.9, s=34)
plt.plot(x_line["X"], y_line, color="crimson", linewidth=2, label="Régression")
plt.xlabel("X")
plt.ylabel("Y")
plt.title("Régression linéaire ajustée")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

In [ ]:
residuals = y_test.to_numpy() - predictions

plt.figure(figsize=(7, 4))
plt.scatter(predictions, residuals, alpha=0.8)
plt.axhline(0, color="crimson", linestyle="--")
plt.xlabel("Valeurs prédites")
plt.ylabel("Résidus")
plt.title("Analyse des résidus sur le jeu de test")
plt.grid(alpha=0.2)
plt.show()